# 13 · Bit Manipulation

*Contest essentials* · 8 exercises

Masks, XOR tricks and enumerating every subset.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ch-13) · [Answer key](https://bloomcode.github.io/cp/python/answer-key.html#ch-13)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 13.1 · Get, set, clear and toggle a bit  `Core`

**Task.** Write `get_bit(x, i)`, `set_bit(x, i)`, `clear_bit(x, i)` and `toggle_bit(x, i)` using `1 << i` with `&`, `|`, `& ~` and `^`.

**Example.** `x = 0b1010`: `get_bit(x, 1)` -> `1`; `set_bit(x, 0)` -> `11`; `clear_bit(x, 3)` -> `2`; `toggle_bit(x, 2)` -> `14`

**Big O.** O(1).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-13-1) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-13-1)

In [ ]:
# 13.1 Get, set, clear and toggle a bit
def get_bit(x, i):
    raise NotImplementedError


def set_bit(x, i):
    raise NotImplementedError


def clear_bit(x, i):
    raise NotImplementedError


def toggle_bit(x, i):
    raise NotImplementedError

In [ ]:
def test_ex_13_1():
    x = 10
    assert get_bit(x, 1) == 1
    assert get_bit(x, 0) == 0
    assert set_bit(x, 0) == 11
    assert clear_bit(x, 3) == 2
    assert toggle_bit(x, 2) == 14


test_ex_13_1()
print('13.1 passed')

### 13.2 · Count set bits three ways  `Core`

**Task.** Write `popcount_str(x)` with `bin(x).count("1")`, `popcount_loop(x)` with Kernighan's `x &= x - 1`, and compare with `int.bit_count()` (Python 3.10+).

**Example.** `popcount_loop(0b101101)` -> `4`; `(255).bit_count()` -> `8`

**Big O.** Kernighan's loop runs once per set bit.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-13-2) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-13-2)

In [ ]:
# 13.2 Count set bits three ways
def popcount_str(x):
    raise NotImplementedError


def popcount_loop(x):
    raise NotImplementedError

In [ ]:
def test_ex_13_2():
    assert popcount_loop(45) == 4 == popcount_str(45)
    assert 255 .bit_count() == 8
    for v in range(300):
        assert popcount_loop(v) == bin(v).count('1') == popcount_str(v)


test_ex_13_2()
print('13.2 passed')

### 13.3 · Power of two  `Core`

**Task.** Write `is_power_of_two(n)` in one expression with `n & (n - 1)`.

**Example.** `is_power_of_two(64)` -> `True`; `is_power_of_two(96)` -> `False`; `is_power_of_two(0)` -> `False`

**Big O.** O(1).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-13-3) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-13-3)

In [ ]:
# 13.3 Power of two
def is_power_of_two(n):
    raise NotImplementedError

In [ ]:
def test_ex_13_3():
    assert is_power_of_two(64)
    assert not is_power_of_two(96)
    assert not is_power_of_two(0)


test_ex_13_3()
print('13.3 passed')

### 13.4 · Enumerate subsets with a bitmask  `Core`

**Task.** Write `all_subsets(items)` looping `mask` from 0 to `2**n - 1` and including item i when bit i of mask is set.

**Example.** `all_subsets(['a', 'b'])` -> `[[], ['a'], ['b'], ['a', 'b']]`

**Big O.** O(2^N * N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-13-4) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-13-4)

In [ ]:
# 13.4 Enumerate subsets with a bitmask
def all_subsets(items):
    raise NotImplementedError

In [ ]:
def test_ex_13_4():
    assert all_subsets(['a', 'b']) == [[], ['a'], ['b'], ['a', 'b']]
    assert len(all_subsets(list(range(5)))) == 32


test_ex_13_4()
print('13.4 passed')

### 13.5 · The lonely number with XOR  `Core`

**Task.** Every number appears twice except one. Write `lonely(nums)` using XOR (`x ^ x == 0`, `x ^ 0 == x`).

**Example.** `lonely([4, 1, 2, 1, 2])` -> `4`

**Big O.** O(N) time, O(1) space.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-13-5) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-13-5)

In [ ]:
# 13.5 The lonely number with XOR
def lonely(nums):
    raise NotImplementedError

In [ ]:
def test_ex_13_5():
    assert lonely([4, 1, 2, 1, 2]) == 4


test_ex_13_5()
print('13.5 passed')

### 13.6 · Lowest set bit  `Core`

**Task.** Write `lowest_bit(x)` returning the value of the lowest set bit with `x & -x`, and `highest_bit_index(x)` with `x.bit_length() - 1`.

**Example.** `lowest_bit(12)` -> `4`; `highest_bit_index(12)` -> `3`

**Big O.** O(1).

**Note.** `x & -x` is the step used by Fenwick trees, which you will meet in contest data structures.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-13-6) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-13-6)

In [ ]:
# 13.6 Lowest set bit
def lowest_bit(x):
    raise NotImplementedError


def highest_bit_index(x):
    raise NotImplementedError

In [ ]:
def test_ex_13_6():
    assert lowest_bit(12) == 4
    assert highest_bit_index(12) == 3


test_ex_13_6()
print('13.6 passed')

### 13.7 · Fixed-width binary strings  `Core`

**Task.** Write `to_bits(x, width)` returning a zero-padded binary string using `format(x, f"0{width}b")`, and `from_bits(s)` using `int(s, 2)`.

**Example.** `to_bits(5, 8)` -> `"00000101"`; `from_bits("1101")` -> `13`

**Big O.** O(width).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-13-7) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-13-7)

In [ ]:
# 13.7 Fixed-width binary strings
def to_bits(x, width):
    raise NotImplementedError


def from_bits(s):
    raise NotImplementedError

In [ ]:
def test_ex_13_7():
    assert to_bits(5, 8) == '00000101'
    assert from_bits('1101') == 13


test_ex_13_7()
print('13.7 passed')

### 13.8 · Subset sum by brute force  `Practice`

**Task.** Write `can_make(nums, target)` that tries every bitmask and returns True if some subset sums to target.

**Example.** `can_make([3, 34, 4, 12, 5, 2], 9)` -> `True`; `can_make([3, 34, 4], 30)` -> `False`

**Big O.** O(2^N * N). Use it to check a smarter solution on small inputs.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-13-8) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-13-8)

In [ ]:
# 13.8 Subset sum by brute force
def can_make(nums, target):
    raise NotImplementedError

In [ ]:
def test_ex_13_8():
    assert can_make([3, 34, 4, 12, 5, 2], 9)
    assert not can_make([3, 34, 4], 30)


test_ex_13_8()
print('13.8 passed')